# 🗡️ Build a Multi-Agent Text Adventure with Redis

**Duration:** ~45 minutes  
**What you'll build:** A Zork-style text adventure game powered by a small multi-agent pipeline.

## Architecture

```
User Command
     │
     ▼
┌─────────────┐      JSON output
│   Router    │ ──────────────────► {intent, params}
│ (RedisVL    │
│  semantic)  │
└─────────────┘
     │
     ▼
┌─────────────┐      reads/writes
│   World     │ ◄────────────────► Redis JSON
│  (Python)   │                    (game state)
└─────────────┘
     │
     ▼
┌─────────────┐      reads/writes
│  Narrator   │ ◄────────────────► Agent Memory Server
│  (LLM call) │                    (conversation history)
└─────────────┘
     │
     ▼
  Response
```

**Redis is doing three distinct jobs here:**
1. **Redis JSON** — stores the game world (locations, exits, inventory) as structured documents you can query and patch in place
2. **RedisVL Semantic Router** — indexes command reference phrases and classifies player input by semantic similarity
3. **Agent Memory Server** — manages the conversation history so the Narrator remembers what happened earlier in the session

No LangChain. No LangGraph. Just Python, OpenAI, and Redis.

---

## Prerequisites

**Before running any cells:**

1. **Get a free Redis Cloud database** → https://redis.io/try-free/  
   Create a free database (includes RedisJSON + RediSearch). Copy the **Public endpoint** and **Password** from the dashboard.

2. **Open RedisInsight** → https://redis.io/insight/ (free desktop app)  
   Connect using your Redis Cloud credentials. You'll watch game state appear as you play.

3. **Start Agent Memory Server locally:**
   ```bash
   pip install agent-memory-server
   agent-memory-server start
   # Starts on http://localhost:8080
   ```

4. **Create a `.env` file** in this directory:
   ```
   OPENAI_API_KEY=sk-...
   REDIS_URL=redis://default:<password>@<host>:<port>
   ```

---
## Part 1 — Setup

In [ ]:
# Install dependencies (run once)
import subprocess, sys
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q",
    "openai", "redis", "redisvl", "requests", "python-dotenv"])
print("✅ Dependencies installed")

In [ ]:
import os, json, uuid, textwrap
import redis
import requests
from openai import OpenAI
from dotenv import load_dotenv

load_dotenv()

# ── Redis connection ─────────────────────────────────────────────────────────
# Reads REDIS_URL from .env — works with Redis Cloud or a local Redis instance.
# Format: redis://default:<password>@<host>:<port>
REDIS_URL = os.getenv("REDIS_URL", "redis://localhost:6379")
r = redis.from_url(REDIS_URL, decode_responses=True)
r.ping()  # will raise if Redis isn't reachable
print("✅ Redis connected:", REDIS_URL.split("@")[-1])  # prints host only, not password

# ── OpenAI client ─────────────────────────────────────────────────────────────
llm = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))
print("✅ OpenAI client ready")

# ── Agent Memory Server URL ───────────────────────────────────────────────────
AMS_URL = os.getenv("AMS_URL", "http://localhost:8080")

---
## Part 2 — Load the World into Redis JSON

The game world lives in Redis as a JSON document. Each location, its exits, and the items it contains are all stored under a single key per game session.

Why Redis JSON instead of a plain Python dict?
- **Persistence** — the game survives a Python crash or kernel restart
- **Atomic path updates** — `JSON.SET game:123 $.current_location '"hallway"'` updates one field without overwriting anything else
- **Queryable** — `JSON.GET game:123 $.locations.entrance.exits` returns just the exits for a location
- **RedisInsight** — you can watch state change in real time in the browser

In [ ]:
def load_world(game_id: str, world_file: str = "data/world.json") -> None:
    """Load a world template into Redis JSON under key game:<game_id>."""
    with open(world_file) as f:
        world = json.load(f)

    # JSON.SET stores the entire document at the root path '$'
    r.json().set(f"game:{game_id}", "$", world)
    print(f"✅ World loaded → key: game:{game_id}")


def get_state(game_id: str) -> dict:
    """Fetch the full game state from Redis JSON."""
    # json().get returns a list when using JSONPath; [0] unwraps it
    return r.json().get(f"game:{game_id}", "$")[0]


def get_current_location(game_id: str) -> dict:
    """Fetch only the current location object — no need to load the whole state."""
    state = get_state(game_id)
    loc_id = state["current_location"]
    return state["locations"][loc_id]


# Create a game session
GAME_ID = str(uuid.uuid4())[:8]  # short ID for readability
load_world(GAME_ID)
print(f"\nGame session: {GAME_ID}")
print("\nOpen RedisInsight, connect to your Redis Cloud database, and search for key:", f"game:{GAME_ID}")

In [ ]:
# Explore the state directly with Redis JSON path queries
key = f"game:{GAME_ID}"

# Where are we?
current_loc_id = r.json().get(key, "$.current_location")[0]
print("Current location ID:", current_loc_id)

# What are the exits from the current location?
exits = r.json().get(key, f"$.locations.{current_loc_id}.exits")[0]
print("Exits:", exits)

# What items are here?
items = r.json().get(key, f"$.locations.{current_loc_id}.items")[0]
print("Items here:", items)

# What's in my inventory?
inventory = r.json().get(key, "$.inventory")[0]
print("Inventory:", inventory)

---
## Part 3 — The Router (RedisVL Semantic Router)

The Router classifies player commands into structured intents using **RedisVL's `SemanticRouter`**. Reference phrases for each intent are embedded and indexed in Redis; incoming commands are matched by semantic similarity — no LLM call required.

Parameter extraction (direction, item name, etc.) is handled with lightweight string parsing after the intent is matched.

In [ ]:
import os
import re

from redisvl.extensions.router import Route, SemanticRouter
from redisvl.utils.vectorize import HFTextVectorizer

os.environ["TOKENIZERS_PARALLELISM"] = "false"

INTENT_ROUTES = [
    Route(
        name="move",
        references=[
            "go north", "go south", "go east", "go west",
            "walk north", "head south", "move east", "run west",
            "north", "south", "east", "west",
        ],
        distance_threshold=0.55,
    ),
    Route(
        name="take",
        references=[
            "take the torch", "pick up the key", "grab the sword",
            "get the item", "take torch", "pick up the old key",
        ],
        distance_threshold=0.55,
    ),
    Route(
        name="drop",
        references=[
            "drop the torch", "put down the key", "leave the sword",
            "drop torch", "put down key",
        ],
        distance_threshold=0.55,
    ),
    Route(
        name="look",
        references=[
            "look", "look around", "describe the room", "what do I see",
            "survey the area", "where am I",
        ],
        distance_threshold=0.55,
    ),
    Route(
        name="inventory",
        references=[
            "inventory", "what am I carrying", "what do I have",
            "check inventory", "show my items",
        ],
        distance_threshold=0.55,
    ),
    Route(
        name="examine",
        references=[
            "examine the torch", "inspect the door", "look at the key",
            "examine door", "inspect the chest",
        ],
        distance_threshold=0.55,
    ),
]

semantic_router = SemanticRouter(
    name="adventure-command-router",
    routes=INTENT_ROUTES,
    vectorizer=HFTextVectorizer(model="sentence-transformers/all-MiniLM-L6-v2"),
    redis_url=REDIS_URL,
    overwrite=True,
)

DIRECTIONS = ("north", "south", "east", "west")
DIRECTION_ALIASES = {"n": "north", "s": "south", "e": "east", "w": "west"}


def _normalize_item(text: str) -> str:
    item = text.lower().strip()
    item = re.sub(r"^(the|a|an)\s+", "", item)
    return item.replace(" ", "_")


def extract_params(intent: str, command: str) -> dict:
    """Fill intent-specific fields after semantic classification."""
    cmd = command.lower().strip()
    result = {"intent": intent}

    if intent == "move":
        tokens = re.findall(r"[a-z]+", cmd)
        for token in tokens:
            direction = DIRECTION_ALIASES.get(token, token)
            if direction in DIRECTIONS:
                result["direction"] = direction
                break

    elif intent in ("take", "drop"):
        for prefix in (
            "pick up the ", "pick up ", "take the ", "take ",
            "drop the ", "drop ", "put down the ", "put down ", "get the ", "get ",
        ):
            if prefix in cmd:
                result["item"] = _normalize_item(cmd.split(prefix, 1)[1])
                break
        if "item" not in result:
            result["item"] = _normalize_item(re.sub(r"^(take|drop|get|pick up|put down)\s+", "", cmd))

    elif intent == "examine":
        for prefix in (
            "examine the ", "examine ", "inspect the ", "inspect ",
            "look at the ", "look at ",
        ):
            if prefix in cmd:
                result["target"] = _normalize_item(cmd.split(prefix, 1)[1])
                break
        if "target" not in result:
            result["target"] = _normalize_item(re.sub(r"^(examine|inspect|look at)\s+", "", cmd))

    return result


def route(command: str) -> dict:
    """Classify a player command into a structured intent dict."""
    match = semantic_router(command)
    if not match.name:
        return {"intent": "unknown"}
    return extract_params(match.name, command)


# Test the router
for cmd in ["go north", "pick up the torch", "look around", "what am I carrying?", "attack the dragon"]:
    intent = route(cmd)
    print(f"{cmd!r:35} → {intent}")

---
## Part 4 — The World Engine

The World Engine is **pure Python** — no LLM needed here. It applies the intent to the game state and returns a plain-English event description (what actually happened) plus the state changes.

This is intentional: deterministic game logic doesn't belong in an LLM. You get speed, zero hallucination risk, and lower cost.

All state mutations go through **Redis JSON path operations**, which update individual fields atomically.

In [ ]:
def handle_world(intent: dict, game_id: str) -> dict:
    """
    Apply an intent to the game world.
    Returns {"success": bool, "event": str, "details": dict}.
    All world mutations are written directly to Redis JSON.
    """
    state = get_state(game_id)
    loc_id = state["current_location"]
    location = state["locations"][loc_id]
    key = f"game:{game_id}"

    action = intent.get("intent")

    # ── LOOK ─────────────────────────────────────────────────────────────────
    if action == "look":
        exits_str = ", ".join(location["exits"].keys()) or "none"
        items_str = ", ".join(location["items"]) or "nothing"
        return {
            "success": True,
            "event": "look",
            "details": {
                "location_name": location["name"],
                "description": location["description"],
                "exits": exits_str,
                "items": items_str,
            },
        }

    # ── MOVE ─────────────────────────────────────────────────────────────────
    elif action == "move":
        direction = intent.get("direction", "").lower()
        destination = location["exits"].get(direction)

        if not destination:
            return {"success": False, "event": f"There is no exit to the {direction}.", "details": {}}

        # Atomic Redis JSON update — only touches current_location
        r.json().set(key, "$.current_location", destination)

        # Track visited locations
        if destination not in state["visited"]:
            r.json().arrappend(key, "$.visited", destination)

        dest_loc = state["locations"][destination]
        return {
            "success": True,
            "event": f"You move {direction}.",
            "details": {
                "location_name": dest_loc["name"],
                "description": dest_loc["description"],
                "exits": ", ".join(dest_loc["exits"].keys()),
                "items": ", ".join(dest_loc["items"]) or "nothing",
                "first_visit": destination not in state["visited"],
            },
        }

    # ── TAKE ─────────────────────────────────────────────────────────────────
    elif action == "take":
        item = intent.get("item", "").lower().replace(" ", "_")
        # Normalize: remove articles
        item = item.replace("the_", "").replace("a_", "")

        if item not in location["items"]:
            return {"success": False, "event": f"There is no {item} here.", "details": {}}

        # Remove from location, add to inventory — two atomic JSON operations
        updated_items = [i for i in location["items"] if i != item]
        r.json().set(key, f"$.locations.{loc_id}.items", updated_items)
        r.json().arrappend(key, "$.inventory", item)

        desc = state["item_descriptions"].get(item, "An unremarkable object.")
        return {"success": True, "event": f"You pick up the {item}.", "details": {"item": item, "description": desc}}

    # ── DROP ─────────────────────────────────────────────────────────────────
    elif action == "drop":
        item = intent.get("item", "").lower().replace(" ", "_")
        item = item.replace("the_", "").replace("a_", "")

        if item not in state["inventory"]:
            return {"success": False, "event": f"You aren't carrying a {item}.", "details": {}}

        updated_inv = [i for i in state["inventory"] if i != item]
        r.json().set(key, "$.inventory", updated_inv)
        r.json().arrappend(key, f"$.locations.{loc_id}.items", item)

        return {"success": True, "event": f"You drop the {item}.", "details": {"item": item}}

    # ── INVENTORY ────────────────────────────────────────────────────────────
    elif action == "inventory":
        inv = state["inventory"]
        carrying = ", ".join(inv) if inv else "nothing"
        return {"success": True, "event": f"You are carrying: {carrying}.", "details": {"inventory": inv}}

    # ── EXAMINE ──────────────────────────────────────────────────────────────
    elif action == "examine":
        target = intent.get("target", "").lower().replace(" ", "_")
        desc = state["item_descriptions"].get(target, f"You see nothing special about the {target}.")
        return {"success": True, "event": f"You examine the {target}.", "details": {"description": desc}}

    # ── UNKNOWN ──────────────────────────────────────────────────────────────
    else:
        return {"success": False, "event": "That's not something you can do here.", "details": {}}


# Quick test
print(handle_world({"intent": "look"}, GAME_ID))

---
## Part 5 — The Narrator Agent + Agent Memory Server

The Narrator turns the dry event output from the World Engine into atmospheric prose. It knows the game's tone and — crucially — **remembers the conversation**.

Memory is handled by [Redis Agent Memory Server](https://redis.github.io/agent-memory-server/), which stores per-session message history in Redis. Every exchange is appended, and we pull the last N messages as context for each LLM call.

**Why AMS instead of just a Python list?**
- Survives kernel restarts and process crashes
- Multiple clients can share the same session (e.g., a web UI + a CLI)
- The memory is stored in Redis — same data store, one less moving part
- Long-term memory mode (not used here) adds semantic search over past sessions

In [ ]:
# ── Agent Memory Server helpers ───────────────────────────────────────────────
# AMS stores working memory as a list of {role, content} messages per session.
# We call it like any REST API — no special SDK required.

def ams_get_messages(session_id: str, window: int = 8) -> list[dict]:
    """Retrieve the last `window` messages from AMS for this session."""
    try:
        resp = requests.get(
            f"{AMS_URL}/v1/memory/working-memory",
            params={"session_id": session_id, "window_size": window},
            timeout=5,
        )
        if resp.status_code == 200:
            data = resp.json()
            return data.get("messages", [])
    except requests.exceptions.ConnectionError:
        pass  # AMS not running — narrator will work without memory
    return []


def ams_add_messages(session_id: str, new_messages: list[dict]) -> None:
    """Append messages to AMS working memory for this session."""
    existing = ams_get_messages(session_id, window=50)
    all_messages = existing + new_messages

    try:
        requests.put(
            f"{AMS_URL}/v1/memory/working-memory",
            json={"session_id": session_id, "messages": all_messages},
            timeout=5,
        )
    except requests.exceptions.ConnectionError:
        pass  # fail silently if AMS is down


# Test AMS connection
try:
    requests.get(f"{AMS_URL}/v1/memory/working-memory", params={"session_id": "test"}, timeout=3)
    print("✅ Agent Memory Server connected")
except Exception:
    print("⚠️  Agent Memory Server not reachable — narrator will work without conversation memory")

In [ ]:
NARRATOR_SYSTEM = """
You are the narrator of a dark, atmospheric text adventure game in the style of Zork.
Your prose is vivid but concise — 2 to 4 sentences per response.
You use second-person present tense ("You see...", "You move...").
Do not invent items, exits, or events beyond what the game engine reports.
Do not include any game mechanics or structured data in your response — pure prose only.
"""


def narrate(command: str, world_result: dict, game_id: str) -> str:
    """
    Turn the world engine's result into atmospheric prose.
    Fetches conversation history from AMS, calls the LLM, then stores the exchange.
    """
    # Pull recent conversation history from Agent Memory Server
    history = ams_get_messages(game_id)

    # Build the prompt: history + current event summary
    event_summary = json.dumps(world_result, indent=2)
    user_message = f'Player command: "{command}"\n\nGame engine result:\n{event_summary}'

    messages = [
        {"role": "system", "content": NARRATOR_SYSTEM},
        *history,
        {"role": "user", "content": user_message},
    ]

    response = llm.chat.completions.create(
        model="gpt-4o-mini",
        messages=messages,
        temperature=0.7,
        max_tokens=200,
    )
    narrative = response.choices[0].message.content.strip()

    # Store this exchange in Agent Memory Server for future turns
    ams_add_messages(game_id, [
        {"role": "user",      "content": user_message},
        {"role": "assistant", "content": narrative},
    ])

    return narrative


# Test the narrator with a manual world result
test_result = {
    "success": True,
    "event": "look",
    "details": {
        "location_name": "Dungeon Entrance",
        "description": "You stand at the entrance to an ancient dungeon. Moss creeps across crumbling stone walls.",
        "exits": "north, east",
        "items": "torch, old_key",
    }
}
print(narrate("look around", test_result, GAME_ID))

---
## Part 6 — Wire It Together

The full pipeline is one function: route → world → narrate.

In [ ]:
def process_turn(command: str, game_id: str) -> str:
    """
    Full agent pipeline for a single player turn.

    1. Router  — RedisVL SemanticRouter classifies command into structured intent
    2. World   — Python applies intent; reads/writes Redis JSON
    3. Narrator — LLM generates prose; reads/writes Agent Memory Server
    """
    intent = route(command)           # Step 1: classify
    result = handle_world(intent, game_id)  # Step 2: apply
    response = narrate(command, result, game_id)  # Step 3: narrate
    return response


# Run a few turns to see everything working together
demo_commands = ["look around", "go north", "look around", "go south", "take the torch"]

print("=" * 60)
for cmd in demo_commands:
    print(f"\n> {cmd}")
    print(textwrap.fill(process_turn(cmd, GAME_ID), width=60))
    print("-" * 60)

**Check RedisInsight** — connect it to your Redis Cloud database and you should see:
- Key `game:<id>` with your current location updated, torch removed from the entrance, added to inventory
- AMS keys storing the conversation history

All the game state is right there, queryable and inspectable.

---
## Part 7 — Play the Game

In [ ]:
# Start a fresh game session for interactive play
PLAY_ID = str(uuid.uuid4())[:8]
load_world(PLAY_ID)

print("=" * 60)
print("  THE FORGOTTEN DUNGEON")
print("  Type 'quit' to exit")
print("=" * 60)
print(textwrap.fill(process_turn("look around", PLAY_ID), width=60))
print()

while True:
    try:
        command = input("> ").strip()
    except (EOFError, KeyboardInterrupt):
        print("\nFarewell, adventurer.")
        break

    if not command:
        continue
    if command.lower() in ("quit", "exit", "q"):
        print("Farewell, adventurer. Your progress is saved in Redis.")
        print(f"Resume with: load_world('{PLAY_ID}')  ← already saved, just re-run process_turn")
        break

    response = process_turn(command, PLAY_ID)
    print(textwrap.fill(response, width=60))
    print()

---
## Part 8 — Checkpoint: What Redis Did

Let's look at the game state after playing a few turns.

In [ ]:
# Inspect the live game state using Redis JSON path queries
key = f"game:{PLAY_ID}"

print("Current location:", r.json().get(key, "$.current_location")[0])
print("Inventory:       ", r.json().get(key, "$.inventory")[0])
print("Locations visited:", r.json().get(key, "$.visited")[0])

# Show items at current location (no need to load the whole state)
loc = r.json().get(key, "$.current_location")[0]
print(f"Items at {loc}:  ", r.json().get(key, f"$.locations.{loc}.items")[0])

In [ ]:
# Inspect conversation history from Agent Memory Server
messages = ams_get_messages(PLAY_ID, window=10)
print(f"Last {len(messages)} messages in AMS for session {PLAY_ID}:")
print()
for msg in messages:
    role = msg['role'].upper()
    content_preview = msg['content'][:80].replace("\n", " ")
    print(f"  [{role}] {content_preview}..." if len(msg['content']) > 80 else f"  [{role}] {msg['content']}")

---
## Part 9 — Extend It (Hackathon Challenge)

Pick one or more of these to build in the remaining time:

### 🥉 Easy — Add a new location
Edit `data/world.json`, add a new entry under `locations`, and wire it into an existing location's `exits`. Re-run `load_world()`.

### 🥈 Medium — Add NPC interaction
Extend the Router with a `"talk"` route (add reference phrases and handle the intent in `handle_world`). Add NPCs to locations in the world JSON. The Narrator already handles free-form events — you just need to pass it the right context.

### 🥇 Hard — Persistent save/load across sessions
Since game state is already in Redis, implement `save_game` (copies a key with an explicit name) and `load_game` (restores it). The game is already persistent — this is about named saves.

### 🏆 Bonus — Use AMS long-term memory
After each session, summarize what happened and store it as a long-term memory in AMS. On a new game, retrieve relevant past memories and give the Narrator "lore" about the dungeon's history.

In [ ]:
# ── CHALLENGE STARTER: Save / Load ────────────────────────────────────────────

def save_game(game_id: str, save_name: str) -> None:
    """Save current game state under a human-readable name."""
    state = get_state(game_id)
    r.json().set(f"save:{save_name}", "$", state)
    print(f"Game saved as '{save_name}'")


def load_game(save_name: str) -> str:
    """Load a saved game, returning a new game_id ready to play."""
    save_data = r.json().get(f"save:{save_name}", "$")
    if not save_data:
        raise ValueError(f"No save found with name '{save_name}'")
    new_id = str(uuid.uuid4())[:8]
    r.json().set(f"game:{new_id}", "$", save_data[0])
    print(f"Loaded '{save_name}' → new game id: {new_id}")
    return new_id


def list_saves() -> list[str]:
    """List all saved games."""
    keys = r.keys("save:*")
    return [k.replace("save:", "") for k in keys]


# Usage:
# save_game(PLAY_ID, "my-first-run")
# resumed_id = load_game("my-first-run")
# process_turn("look around", resumed_id)

---
## Summary

You built a multi-agent text adventure game where:

| Component | What it does | Technology |
|-----------|-------------|------------|
| **Router** | Classifies commands → structured intent | **RedisVL SemanticRouter** |
| **World Engine** | Applies logic, validates moves | Pure Python + **Redis JSON** |
| **Narrator** | Generates atmospheric prose | OpenAI gpt-4o-mini + **Agent Memory Server** |

**Redis is doing the heavy lifting:**
- `JSON.SET` / `JSON.GET` — structured game world stored and queried with path syntax
- `JSON.ARRAPPEND` — atomic item pickup / inventory updates
- **RedisVL Semantic Router** — command intent classification via vector search over reference phrases
- **Agent Memory Server** — per-session conversation history that survives restarts and can be searched

No LangChain. No LangGraph. ~200 lines of Python.